# Laboratorio Dirigido N.° 06
## Semana 6: Limpieza de Datos — Pipeline sobre `ventas_original`

| Campo | Detalle |
|---|---|
| **Curso** | Fundamentos de Gestión de Datos |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Semana** | 6 |
| **Caso** | Abarrotes Esperanza |
| **Tabla de trabajo** | `ventas_original` |
| **Entregable** | Notebook ejecutado con respuestas completas |

> **Instrucción:** Este notebook trabaja directamente desde `ventas_original`. No asumas que existen tablas normalizadas. Ejecuta todas las celdas en orden, completa cada respuesta y exporta el dataset limpio al finalizar.

## 📋 Revisión de conocimientos de la semana

Antes de ejecutar el código, completa el siguiente cuadro con tus propias palabras.

| N.° | Concepto / Principio | Definición (con tus propias palabras) |
|---|---|---|
| 1 | Pipeline de limpieza de datos | |
| 2 | Valores faltantes e imputación | |
| 3 | Outliers y método IQR | |
| 4 | Normalización Min-Max | |
| 5 | Tabla desnormalizada | |
| 6 | Variabilidad | |
| 7 | Mediana | |
| 8 | Desviación estándar | |
| 9 | Cuartiles Q2 y Q3 | |
| 10 | Rango intercuartil (IQR) | |
| 11 | Técnicas de imputación | |

> **Nota teórica — Técnicas de imputación:** cuando una columna tiene valores ausentes se puede completar con la **media** (distribución simétrica), la **mediana** (si hay outliers), la **moda** (variables categóricas), un valor constante, forward/backward fill (series de tiempo) o modelos como KNN. En este laboratorio: **mediana** para numéricas y **moda** para categóricas.


---
## Actividad 1. Carga de librerías y base de datos desde GitHub

In [ ]:
try:
    import requests
except ModuleNotFoundError:
    requests = None

import sqlite3
import pandas as pd
import numpy as np
import warnings
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError:
    plt = None
    sns = None

url = "https://raw.githubusercontent.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos/main/casos/16_abarrotes_ventas_inventario/abarrotes_ventas_inventario.db"

if requests is not None:
    r = requests.get(url)
    r.raise_for_status()
    contenido = r.content
else:
    from urllib.request import urlopen
    with urlopen(url) as respuesta:
        contenido = respuesta.read()

open("abarrotes_ventas_inventario.db", "wb").write(contenido)

conn = sqlite3.connect("abarrotes_ventas_inventario.db")

df = pd.read_sql_query("SELECT * FROM ventas_original LIMIT 5", conn)
df


**Pregunta 1. ¿Por qué en la Semana 6 se trabaja desde `ventas_original` y no desde tablas normalizadas?**

> **Respuesta:**

---
## Actividad 2. Carga completa de `ventas_original`

In [ ]:
ventas_original = pd.read_sql_query("SELECT * FROM ventas_original;", conn)
ventas_original.head()


**Pregunta 2. ¿Qué columnas presentan mayor riesgo de calidad y por qué?**

> **Respuesta:**

---
## Actividad 3. Diagnóstico inicial de calidad

In [ ]:
print("Filas y columnas:", ventas_original.shape)
display(ventas_original.info())

diagnostico = pd.DataFrame({
    "tipo_dato": ventas_original.dtypes.astype(str),
    "nulos": ventas_original.isna().sum(),
    "porcentaje_nulos": (ventas_original.isna().mean() * 100).round(2),
    "unicos": ventas_original.nunique(dropna=True)
})
diagnostico.sort_values("porcentaje_nulos", ascending=False)


**Pregunta 3. ¿Qué señales muestran que la tabla está desnormalizada?**

> **Respuesta:**

---
## Actividad 4. Revisión de duplicados y consistencia básica

In [ ]:
duplicados = ventas_original.duplicated().sum()
print("Duplicados exactos:", duplicados)

columnas_texto = ventas_original.select_dtypes(include="object").columns
resumen_texto = ventas_original[columnas_texto].nunique().sort_values(ascending=False)
resumen_texto


**Pregunta 4. ¿Qué problemas pueden aparecer si no se corrigen formatos de texto, fecha y número?**

> **Respuesta:**

---
## 📊 Tabla de conocimientos — conceptos adicionales

| N.° | Concepto | Definición (con tus propias palabras) |
|---|---|---|
| 6 | Variabilidad | |
| 7 | Mediana | |
| 8 | Desviación estándar | |
| 9 | Cuartiles Q2 y Q3 | |
| 10 | Rango intercuartil (IQR) | |
| 11 | Técnicas de imputación | |

> **Nota teórica — Técnicas de imputación:** cuando una columna tiene valores ausentes se puede completar con la **media** (distribución simétrica), la **mediana** (preferible si hay outliers), la **moda** (variables categóricas o discretas), un valor constante, imputación forward/backward (series de tiempo) o modelos predictivos como KNN. En este laboratorio usamos **mediana** para numéricas y **moda** para categóricas.


---
## Actividad 4a. Estadísticas descriptivas e histogramas


In [ ]:
# Estadísticas descriptivas
ventas_original.describe().round(2)

In [ ]:
# Variables numéricas a visualizar
cols_num = [c for c in [
    "precio_venta_unitario", "costo_unitario", "cantidad_unidades",
    "monto_venta_soles", "margen_venta_soles"
] if c in ventas_original.columns]

# Histogramas
fig, axes = plt.subplots(len(cols_num), 1, figsize=(10, 4*len(cols_num)))
if len(cols_num) == 1: axes = [axes]
for ax, col in zip(axes, cols_num):
    ventas_original[col].dropna().hist(ax=ax, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(f"Histograma — {col}", fontsize=11)
    ax.set_xlabel(col); ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

> **Pregunta 4a.** ¿Qué distribución predomina en las variables numéricas? ¿Hay alguna con alto sesgo?

> **Respuesta:**


---
## Actividad 4b. Diagramas de cajas — detección visual de outliers


In [ ]:
# Boxplots — los puntos fuera de los bigotes son outliers
fig, axes = plt.subplots(1, len(cols_num), figsize=(4*len(cols_num), 5))
if len(cols_num) == 1: axes = [axes]
for ax, col in zip(axes, cols_num):
    ventas_original.boxplot(column=col, ax=ax)
    ax.set_title(col, fontsize=9)
plt.suptitle("Boxplots — variables continuas (puntos = outliers)", y=1.02)
plt.tight_layout()
plt.show()

> **Pregunta 4b.** ¿Cuáles variables muestran más outliers en el boxplot? ¿Cómo se relaciona esto con el método IQR de la Actividad 9?

> **Respuesta:**


---
## Actividad 4c. Gráficas de barras y diagrama de pie


In [ ]:
# Variables categóricas con ≤ 15 valores únicos
cols_cat = [c for c in ventas_original.select_dtypes(include="object").columns
            if ventas_original[c].nunique() <= 15]

# Gráficas de barras
fig, axes = plt.subplots(len(cols_cat), 1, figsize=(10, 4*len(cols_cat)))
if len(cols_cat) == 1: axes = [axes]
for ax, col in zip(axes, cols_cat):
    counts = ventas_original[col].value_counts().head(10)
    counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Distribución — {col}", fontsize=11)
    ax.set_ylabel("Frecuencia"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Diagrama de pie — primera variable categórica
if cols_cat:
    col_pie = cols_cat[0]
    counts = ventas_original[col_pie].value_counts().head(8)
    fig, ax = plt.subplots(figsize=(7, 5))
    counts.plot.pie(ax=ax, autopct="%1.1f%%", startangle=90)
    ax.set_title(f"Proporción — {col_pie}", fontsize=12)
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

> **Pregunta 4c.** ¿Qué categoría predomina y qué porcentaje del total representa?

> **Respuesta:**


---
## Actividad 4d. Tablas cruzadas


In [ ]:
# Tabla cruzada entre dos variables categóricas
if len(cols_cat) >= 2:
    tabla_cruzada = pd.crosstab(
        ventas_original[cols_cat[0]],
        ventas_original[cols_cat[1]],
        margins=True
    )
    print("=== TABLA CRUZADA ===")
    display(tabla_cruzada)

    # Mapa de calor
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(tabla_cruzada.iloc[:-1, :-1], annot=True, fmt='d',
                cmap="Blues", ax=ax)
    ax.set_title("Mapa de calor — Tabla cruzada")
    plt.tight_layout()
    plt.show()

> **Pregunta 4d.** ¿Qué relación encontraron entre las dos variables categóricas del crosstab?

> **Respuesta:**


---
## Actividad 5. Funciones de limpieza

In [ ]:
def limpiar_texto(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

def convertir_numero(serie):
    return pd.to_numeric(serie, errors="coerce")

def convertir_fecha_mixta(serie):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        fecha_1 = pd.to_datetime(serie, errors="coerce", dayfirst=False)
        fecha_2 = pd.to_datetime(serie, errors="coerce", dayfirst=True)
    return fecha_1.fillna(fecha_2)


**Pregunta 5. Justifique una decisión de imputación aplicada en el laboratorio.**

> **Respuesta:**

---
## Actividad 6. Corrección de tipos y formatos

In [ ]:
df_limpio = ventas_original.copy()

for col in df_limpio.select_dtypes(include="object").columns:
    if "fecha" in col.lower():
        df_limpio[col] = convertir_fecha_mixta(df_limpio[col])
    else:
        df_limpio[col] = limpiar_texto(df_limpio[col])

columnas_numericas = [
    "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
    "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
    "monto_venta_soles", "margen_venta_soles", "productos_costo_unitario",
    "productos_precio_lista"
]

for col in columnas_numericas:
    if col in df_limpio.columns:
        df_limpio[col] = convertir_numero(df_limpio[col])

df_limpio.dtypes


**Pregunta 6. ¿Qué diferencia hay entre eliminar duplicados y corregir inconsistencias de contenido?**

> **Respuesta:**

---
## Actividad 7. Imputación de valores faltantes

In [ ]:
nulos_antes = df_limpio.isna().sum()

for col in df_limpio.select_dtypes(include=["number"]).columns:
    mediana = df_limpio[col].median()
    df_limpio[col] = df_limpio[col].fillna(mediana)

for col in df_limpio.select_dtypes(include=["string", "object"]).columns:
    moda = df_limpio[col].mode(dropna=True)
    valor = moda.iloc[0] if len(moda) > 0 else "Sin Dato"
    df_limpio[col] = df_limpio[col].fillna(valor)

for col in df_limpio.select_dtypes(include=["datetime64[ns]"]).columns:
    mediana_fecha = df_limpio[col].dropna().median()
    if pd.isna(mediana_fecha):
        mediana_fecha = pd.Timestamp("1900-01-01")
    df_limpio[col] = df_limpio[col].fillna(mediana_fecha)

nulos_despues = df_limpio.isna().sum()

pd.DataFrame({
    "nulos_antes": nulos_antes,
    "nulos_despues": nulos_despues,
    "nulos_corregidos": nulos_antes - nulos_despues
}).query("nulos_antes > 0 or nulos_despues > 0")


**Pregunta 7. ¿Por qué se usa IQR para detectar outliers y cuándo no conviene eliminar registros?**

> **Respuesta:**

---
## Actividad 8. Tratamiento de duplicados

In [ ]:
filas_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
filas_despues = len(df_limpio)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Duplicados eliminados:", filas_antes - filas_despues)


**Pregunta 8. ¿Para qué sirve normalizar variables numéricas en una etapa posterior de análisis?**

> **Respuesta:**

---
## Actividad 9. Detección y tratamiento de outliers con IQR

In [ ]:
columnas_outliers = [
    col for col in [
        "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
        "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
        "monto_venta_soles", "margen_venta_soles", "productos_costo_unitario",
        "productos_precio_lista"
    ]
    if col in df_limpio.columns
]

reporte_outliers = []

for col in columnas_outliers:
    q1 = df_limpio[col].quantile(0.25)
    q3 = df_limpio[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    mascara = (df_limpio[col] < lim_inf) | (df_limpio[col] > lim_sup)
    reporte_outliers.append([col, int(mascara.sum()), lim_inf, lim_sup])
    df_limpio[col] = df_limpio[col].astype(float).clip(lower=lim_inf, upper=lim_sup)

pd.DataFrame(reporte_outliers, columns=["columna", "outliers_detectados", "limite_inferior", "limite_superior"])


**Pregunta 9. Interprete el reporte antes/después: ¿la calidad del dataset mejoró? Sustente.**

> **Respuesta:**

---
## Actividad 10. Normalización Min-Max

In [ ]:
df_normalizado = df_limpio.copy()

for col in columnas_outliers:
    minimo = df_normalizado[col].min()
    maximo = df_normalizado[col].max()
    if maximo != minimo:
        df_normalizado[col + "_minmax"] = (df_normalizado[col] - minimo) / (maximo - minimo)
    else:
        df_normalizado[col + "_minmax"] = 0

df_normalizado[[col for col in df_normalizado.columns if col.endswith("_minmax")]].describe().round(3)


**Pregunta 10. ¿Qué tabla queda creada en SQLite y cómo se podría usar en el siguiente laboratorio?**

> **Respuesta:**

---
## Actividad 11. Reporte antes/después

In [ ]:
reporte_final = pd.DataFrame({
    "metrica": ["filas", "columnas", "nulos_totales", "duplicados_exactos"],
    "antes": [
        ventas_original.shape[0],
        ventas_original.shape[1],
        int(ventas_original.isna().sum().sum()),
        int(ventas_original.duplicated().sum())
    ],
    "despues": [
        df_normalizado.shape[0],
        df_normalizado.shape[1],
        int(df_normalizado.isna().sum().sum()),
        int(df_normalizado.duplicated().sum())
    ]
})

reporte_final


**Pregunta 11. ¿Cómo demostrarías que la limpieza mejoró la calidad del dataset? ¿Qué métricas son las más relevantes?**

> **Respuesta:**

---
## Actividad 12. Exportación del dataset limpio

In [ ]:
df_normalizado.to_sql("ventas_original_limpia_s6", conn, if_exists="replace", index=False)
df_normalizado.to_csv("ventas_original_limpia_s6.csv", index=False, encoding="utf-8-sig")

validacion = pd.read_sql_query("SELECT COUNT(*) AS filas_limpias FROM ventas_original_limpia_s6;", conn)
validacion


**Pregunta 12. ¿Por qué es importante guardar el dataset limpio como tabla nueva en lugar de reemplazar la original?**

> **Respuesta:**

---
## Conclusiones

**Conclusión 1:**

> 

**Conclusión 2:**

> 

**Conclusión 3:**

>

---
## Rúbrica holística de evaluación

| Criterio | Excelente | Bueno | Requiere mejora | No aceptable | Puntaje |
|---|---|---|---|---|:---:|
| **1. Carga y diagnóstico** (Act. 1–3) | Carga desde GitHub, muestra shape/info, tabla de nulos ordenada y detecta duplicados. | Carga la base y presenta diagnóstico básico con alguna observación faltante. | Carga la base pero el diagnóstico es incompleto o sin ordenar. | No carga la base o no presenta diagnóstico. | /4 |
| **2. Limpieza e imputación** (Act. 4–7) | Aplica las tres funciones, imputa nulos con mediana/moda, verifica antes/después correctamente. | Aplica la mayoría de funciones con alguna imperfección menor en la imputación. | Aplica parcialmente las funciones o la imputación presenta errores. | No aplica funciones de limpieza ni imputa nulos. | /4 |
| **3. Duplicados y outliers** (Act. 8–9) | Elimina duplicados, aplica IQR con clip y genera reporte completo de outliers. | Elimina duplicados y detecta outliers, pero el reporte es parcial. | Solo trata duplicados o solo detecta outliers, sin completar ambos pasos. | No trata duplicados ni outliers. | /4 |
| **4. Normalización y exportación** (Act. 10–12) | Aplica Min-Max, verifica rango [0,1] y exporta a SQLite y CSV con SELECT de validación. | Normaliza y exporta correctamente, con alguna observación menor. | Normalización o exportación incompleta o sin validación. | No normaliza ni exporta el dataset limpio. | /4 |
| **5. Preguntas reflexivas** (Preg. 1–12) | Las 12 preguntas respondidas con argumentos técnicos precisos y justificación clara. | Al menos 9 preguntas con argumentos aceptables. | Menos de 9 preguntas respondidas o respuestas superficiales. | No responde las preguntas o deja espacios vacíos. | /4 |
| | | | | **TOTAL** | **/20** |

---
*Laboratorio Dirigido N.° 06 — Fundamentos de Gestión de Datos — Pilar Rocío Sayán Mejía*